# Task 9: Anchor-free detector loss (CIoU box regression + classification)

In [1]:
import torch
import math


In [2]:
def box_iou(box1, box2):
    x1 = torch.max(box1[...,0], box2[...,0])
    y1 = torch.max(box1[...,1], box2[...,1])
    x2 = torch.min(box1[...,2], box2[...,2])
    y2 = torch.min(box1[...,3], box2[...,3])
    inter = (x2-x1).clamp(0) * (y2-y1).clamp(0)
    area1 = (box1[...,2]-box1[...,0])*(box1[...,3]-box1[...,1])
    area2 = (box2[...,2]-box2[...,0])*(box2[...,3]-box2[...,1])
    union = area1 + area2 - inter + 1e-7
    return inter/union


In [3]:
def ciou_loss(pred, target):
    iou = box_iou(pred, target)

    px = (pred[...,0]+pred[...,2])/2
    py = (pred[...,1]+pred[...,3])/2
    tx = (target[...,0]+target[...,2])/2
    ty = (target[...,1]+target[...,3])/2
    center_dist = (px-tx)**2 + (py-ty)**2

    ex1 = torch.min(pred[...,0], target[...,0])
    ey1 = torch.min(pred[...,1], target[...,1])
    ex2 = torch.max(pred[...,2], target[...,2])
    ey2 = torch.max(pred[...,3], target[...,3])
    diag = (ex2-ex1)**2 + (ey2-ey1)**2 + 1e-7

    w_p = pred[...,2]-pred[...,0]
    h_p = pred[...,3]-pred[...,1]
    w_t = target[...,2]-target[...,0]
    h_t = target[...,3]-target[...,1]
    v = (4/math.pi**2) * (torch.atan(w_t/h_t) - torch.atan(w_p/h_p))**2
    with torch.no_grad():
        alpha = v/(1-iou+v+1e-7)

    ciou = iou - center_dist/diag - alpha*v
    return (1-ciou).mean()

def cls_loss(pred_logits, target_cls):
    return torch.nn.functional.binary_cross_entropy_with_logits(pred_logits, target_cls)


In [4]:
pred_boxes = torch.tensor([[10,10,50,50],[20,20,60,80]], dtype=torch.float32, requires_grad=True)
target_boxes = torch.tensor([[12,12,48,52],[22,18,58,78]], dtype=torch.float32)

loss = ciou_loss(pred_boxes, target_boxes)
loss.backward()
print("ciou loss:", loss.item())

pred_cls = torch.randn(2, 3, requires_grad=True)
target_cls = torch.tensor([[1,0,0],[0,1,0]], dtype=torch.float32)
print("cls loss:", cls_loss(pred_cls, target_cls).item())


ciou loss: 0.16954699158668518
cls loss: 0.5541992783546448
